# Toxic Comment Classification — Preprocessing

## Goal
The goal of this notebook is to clean and prepare toxic comment text data before modeling.

## What this notebook does
- explores raw text
- removes noisy patterns such as URLs, HTML, IP addresses, and Wikipedia-specific artifacts
- normalizes obfuscated profanity
- creates additional text-based numerical features
- prepares train and test data for later modeling

## Why preprocessing matters
Raw comments often contain irrelevant noise and inconsistent writing styles.  
Preprocessing helps make the data cleaner, more informative, and easier for machine learning models to learn from.

In [ ]:
# Importing libraries
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet')

In [ ]:
# loading the data
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")
test_labels_df = pd.read_csv("/content/test_labels.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Test labels shape:", test_labels_df.shape)

In [ ]:
# Raw data(before)
train_df.head()

## Raw Text Examples

Before preprocessing, comments may contain:
- URLs
- HTML tags
- Wikipedia formatting artifacts
- obfuscated profanity
- repeated characters
- inconsistent punctuation and casing

These patterns add noise and make learning harder for the model.

In [ ]:
# Ram comment example
sample_raw = train_df["comment_text"].dropna().sample(5, random_state=42).tolist()

for i, text in enumerate(sample_raw, 1):
    print(f"Example {i}:")
    print(text)
    print("-" * 100)

In [ ]:
# Preprocessing functions
lemmatizer = WordNetLemmatizer()

def remove_ip_addresses(text):
    """Remove IP addresses like 192.168.0.1"""
    return re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '', text)

def remove_urls(text):
    """Remove URLs"""
    return re.sub(r'http\S+|www\S+', '', text)

def remove_html(text):
    """Remove HTML tags"""
    return re.sub(r'<.*?>', '', text)

def remove_wikipedia_artifacts(text):
    """Remove Wikipedia-specific noise"""
    return re.sub(r'\b(wikipedia|talk|user:).*$', '', text, flags=re.IGNORECASE)

def normalize_profanity(text):
    """Restore obfuscated profanity to consistent form"""
    patterns = {
        r'f[\W_]*u[\W_]*c[\W_]*k+': 'fuck',
        r's[\W_]*h[\W_]*i[\W_]*t+': 'shit',
        r'b[\W_]*i[\W_]*t[\W_]*c[\W_]*h+': 'bitch',
        r'a[\W_]*s[\W_]*s+': 'ass',
        r'd[\W_]*i[\W_]*c[\W_]*k+': 'dick',
        r'p[\W_]*u[\W_]*s[\W_]*s[\W_]*y+': 'pussy',
        r'c[\W_]*u[\W_]*n[\W_]*t+': 'cunt',
        r'n[\W_]*i[\W_]*g[\W_]*g[\W_]*e[\W_]*r+': 'nigger',
        r'f[\W_]*a[\W_]*g+': 'fag',
        r's[\W_]*l[\W_]*u[\W_]*t+': 'slut',
        r'w[\W_]*h[\W_]*o[\W_]*r[\W_]*e+': 'whore',
        r'b[\W_]*a[\W_]*s[\W_]*t[\W_]*a[\W_]*r[\W_]*d+': 'bastard'
    }
    for pattern, replacement in patterns.items():
        text = re.sub(pattern, replacement, text)
    return text

def normalize_repeated_chars(text):
    """Reduce repeated characters like loooool -> loool"""
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def remove_non_ascii(text):
    """Remove emojis and non-ASCII characters"""
    return text.encode('ascii', 'ignore').decode()

def remove_numbers(text):
    """Remove numbers"""
    return re.sub(r'\d+', '', text)

def clean_punctuation(text):
    """Remove punctuation except ! and ?"""
    return re.sub(r'[^\w\s!?]', '', text)

def remove_extra_spaces(text):
    """Remove extra whitespace"""
    return re.sub(r'\s+', ' ', text).strip()

def remove_stopwords(text):
    """Remove common English stopwords"""
    return " ".join([w for w in text.split() if w not in ENGLISH_STOP_WORDS])

def lemmatize_text(text):
    """Convert words to their base form"""
    return " ".join([lemmatizer.lemmatize(w) for w in text.split()])

def clean_text(text):
    """
    Full text cleaning pipeline:
    - lowercase
    - remove IPs, URLs, HTML, Wikipedia artifacts
    - normalize profanity
    - clean punctuation
    - remove extra spaces
    """
    text = str(text).lower()
    text = remove_ip_addresses(text)
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_wikipedia_artifacts(text)
    text = normalize_profanity(text)
    text = clean_punctuation(text)
    text = remove_extra_spaces(text)
    return text

def add_text_features(df, text_column='comment_text'):
    """
    Add numerical text features
    """
    df = df.copy()
    df['comment_length'] = df[text_column].astype(str).apply(len)
    df['word_count'] = df[text_column].astype(str).apply(lambda x: len(x.split()))
    df['num_exclamation'] = df[text_column].astype(str).apply(lambda x: x.count('!'))
    df['num_question'] = df[text_column].astype(str).apply(lambda x: x.count('?'))
    df['num_uppercase'] = df[text_column].astype(str).apply(lambda x: sum(1 for c in x if c.isupper()))
    return df

def preprocess_train(df, text_column='comment_text'):
    """
    Preprocess training data:
    - extract features before cleaning
    - clean text
    """
    df = add_text_features(df, text_column=text_column)
    df[text_column] = df[text_column].astype(str).apply(clean_text)
    return df

def remove_unlabeled_rows(labels: pd.DataFrame, test: pd.DataFrame):
    """
    Remove rows with -1 labels from test labels and test data
    """
    LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    mask = (labels[LABELS] == -1).any(axis=1)
    return labels[~mask].reset_index(drop=True), test[~mask].reset_index(drop=True)

def preprocess_test(test, labels, text_column='comment_text'):
    """
    Preprocess test data
    """
    labels_cleaned, test_cleaned = remove_unlabeled_rows(labels, test)
    test_cleaned = add_text_features(test_cleaned, text_column=text_column)
    test_cleaned[text_column] = test_cleaned[text_column].astype(str).apply(clean_text)
    return test_cleaned, labels_cleaned

In [ ]:
# before/after examples
demo_samples = train_df["comment_text"].dropna().sample(5, random_state=1).tolist()

for i, text in enumerate(demo_samples, 1):
    print(f"Original {i}:")
    print(text)
    print()
    print(f"Cleaned {i}:")
    print(clean_text(text))
    print("=" * 120)

## Why these preprocessing steps were chosen

The cleaning pipeline focuses on removing noise while preserving useful meaning.

### Removed
- IP addresses
- URLs
- HTML tags
- Wikipedia-specific artifacts

### Normalized
- obfuscated profanity such as `f***` into a consistent form
- punctuation noise
- excessive spaces

### Preserved
- exclamation marks and question marks, because they can reflect intensity or aggression

In [ ]:
# applying preprocessing
train_clean = preprocess_train(train_df)
test_clean, test_labels_clean = preprocess_test(test_df, test_labels_df)

print("Processed train shape:", train_clean.shape)
print("Processed test shape:", test_clean.shape)
print("Processed test labels shape:", test_labels_clean.shape)

In [ ]:
# raw vs cleaned text length
raw_lengths = train_df["comment_text"].astype(str).apply(len)
clean_lengths = train_clean["comment_text"].astype(str).apply(len)

plt.figure(figsize=(8, 5))
plt.hist(raw_lengths, bins=50, alpha=0.6, label="Raw")
plt.hist(clean_lengths, bins=50, alpha=0.6, label="Cleaned")
plt.xlabel("Comment length")
plt.ylabel("Frequency")
plt.title("Raw vs Cleaned Comment Length Distribution")
plt.legend()
plt.show()

In [ ]:
# label distribution
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
label_counts = train_clean[LABELS].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
label_counts.plot(kind="bar")
plt.title("Label Distribution in Training Data")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# engineered feature distribution
feature_cols = ['comment_length', 'word_count', 'num_exclamation', 'num_question', 'num_uppercase']
train_clean[feature_cols].describe()

In [ ]:
# Visualize engineered features
for col in feature_cols:
    plt.figure(figsize=(7, 4))
    plt.hist(train_clean[col], bins=40)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# toxic vs non-toxic comparison
train_clean["is_toxic_any"] = train_clean[LABELS].sum(axis=1).apply(lambda x: 1 if x > 0 else 0)

feature_means = train_clean.groupby("is_toxic_any")[feature_cols].mean().T
feature_means.columns = ["Non-toxic", "Toxic"]

feature_means.plot(kind="bar", figsize=(10, 5))
plt.title("Average Text Features: Toxic vs Non-toxic Comments")
plt.ylabel("Average value")
plt.xticks(rotation=45)
plt.show()

## Interpretation of engineered features

The additional numerical features help capture *how* a comment is written, not only *what* words it contains.

For example:
- `comment_length` may reflect long aggressive messages
- `num_exclamation` may capture emotional intensity
- `num_question` may reflect confrontational style
- `num_uppercase` may indicate shouting or strong emphasis

These features can later complement text-based representations such as TF-IDF.

In [ ]:
# save cleaned data
train_clean.to_csv("/content/train_clean.csv", index=False)
test_clean.to_csv("/content/test_clean.csv", index=False)
test_labels_clean.to_csv("/content/test_labels_clean.csv", index=False)

print("Cleaned files saved.")

# Conclusion

In this notebook, the raw toxic comment data was cleaned and enriched with additional text features.

## Main outcomes
- noisy elements such as URLs, HTML, IP addresses, and Wikipedia artifacts were removed
- obfuscated profanity was normalized
- punctuation-based and style-based numerical features were added
- train and test data were prepared for future modeling

## Why this helps later
This preprocessing stage creates a cleaner and more structured input for future models, whether they are baseline machine learning models or more advanced neural architectures.

In [ ]:
summary = pd.DataFrame({
    "Dataset": ["Train", "Test", "Test Labels"],
    "Rows": [len(train_clean), len(test_clean), len(test_labels_clean)],
    "Columns": [train_clean.shape[1], test_clean.shape[1], test_labels_clean.shape[1]]
})

summary